In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt

import openai
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# data

In [ ]:
config = load_config("configs/config_finetune_merfish.yaml")
config.data_name = "MERFISH_26"
config.refresh_paths()
name_truth = config.name_truth

In [ ]:
# --- Load data ---
data_path = str(dataset_file("merfish", config.data_name))
adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]

sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)


# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))



In [ ]:
# if graph type is countplusgene
# Initialize a dictionary to store the top genes per cell type
top_genes_per_cell_type = {}

for cell_type in adata.obs['cell_type'].unique():
    # Subset the data for the current cell type
    adata_subset = adata[adata.obs['cell_type'] == cell_type].copy()
    if len(adata_subset) < 30:
        continue

    
    # Compute highly variable genes within the subset
    sc.pp.highly_variable_genes(
        adata_subset,
        n_top_genes=5,
        flavor='seurat',
        subset=False,
        layer=None,
        inplace=True
    )
    
    # Retrieve the top 5 highly variable genes
    top_genes = adata_subset.var.loc[adata_subset.var['highly_variable'], :].index.tolist()
    
    # Store the results in the dictionary
    top_genes_per_cell_type[cell_type] = top_genes

# Convert the dictionary to a DataFrame for better visualization
top_genes_df = pd.DataFrame.from_dict(top_genes_per_cell_type, orient='index').transpose()

# get all the top genes
top_genes = list(set(top_genes_df.values.flatten()))

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=top_genes)



## sample data

In [ ]:
# 设定随机种子
seed = 42  # 你可以根据需要修改这个值

# 定义分割比例 p (比如 0.7 表示 70% 数据用于训练，30% 数据用于测试)
p = config.prototype_p

# 分割数据集为训练集和测试集
train_neighbor_normalized_df, val_neighbor_normalized_df = train_test_split(neighbor_normalized_df, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


train_neighbor_normalized_df_genes, val_neighbor_normalized_df_genes = train_test_split(neighbor_normalized_df_genes, 
                                                            test_size=1-p, 
                                                            random_state=seed,
                                                            stratify=adata.obs[config.name_truth]
                                                           )


In [ ]:
# check sample distribution
adata.obs[config.name_truth].loc[train_neighbor_normalized_df.index].value_counts()

In [ ]:
# validation in finetune is not necessary
# # finetune train and val data
# _, val_for_finetune = train_test_split(val_neighbor_normalized_df, 
#                                                             test_size=0.1, 
#                                                             random_state=seed
#                                                            )

# adata.obs.loc[val_for_finetune.index, config.name_truth].value_counts()

# prompt

In [ ]:
unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes





In [ ]:
prompt.finetune_system_celltype_geneorder(config)


In [ ]:
prompt.finetune_user_celltype_geneorder(train_neighbor_normalized_df, train_neighbor_normalized_df_genes, 0, config)

In [ ]:
prompt.finetune_assistant(train_neighbor_normalized_df, 0, adata.obs[config.name_truth])

# GPT-4o-mini

In [ ]:
print(f"finetune_json/{config.data_name}_{config.model_type}/")

In [ ]:
# generate json for finetune
output_folder = f"finetune_json/{config.data_name}_{config.model_type}/"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

train_output_file = f"{output_folder}{config.data_name}_train_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
print(f"Generating json for training into {train_output_file}")
with open(train_output_file, 'w') as f:
    for i in range(train_neighbor_normalized_df.shape[0]):
        system_p = prompt.finetune_system_celltype_geneorder(config)
        user_p = prompt.finetune_user_celltype_geneorder(train_neighbor_normalized_df, train_neighbor_normalized_df_genes, i, config)
        assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
        
        row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
        json_str = json.dumps(row_data)
        f.write(json_str + '\n')  

# val_output_file = f"{output_folder}{config.data_name}_val_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.json"
# print(f"Generating json for validation into {val_output_file}")
# with open(val_output_file, 'w') as f:
#     for i in range(val_for_finetune.shape[0]):
#         system_p = prompt.finetune_system_deconv(config)
#         user_p = prompt.finetune_user_deconv(val_for_finetune, i, config)
#         assistant_p = prompt.finetune_assistant(val_for_finetune, i, adata.obs[config.name_truth])

#         row_data = {"messages": [{"role": "system", "content": system_p}, {"role": "user", "content": user_p}, {"role": "assistant", "content": assistant_p}]}
#         json_str = json.dumps(row_data)
#         f.write(json_str + '\n')  

In [ ]:
print(system_p + user_p + assistant_p)

In [ ]:
train_file = client.files.create(
  file=open(train_output_file, "rb"),
  purpose="fine-tune"
)

# val_file = client.files.create(
#   file=open(val_output_file, "rb"),
#   purpose="fine-tune"
# )

finetune_job = client.fine_tuning.jobs.create(
  training_file=train_file.id,
  # validation_file=val_file.id,
  model="gpt-4o-mini-2024-07-18",
  suffix=config.model_type  # default is personal
)


# gemini 1.5 flash

In [ ]:
import google

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["API_KEY"])
for model_info in genai.list_tuned_models():
    print(model_info.name)

In [ ]:

tunable_models = [
    m for m in genai.list_models()
    if "createTunedModel" in m.supported_generation_methods]
tunable_models

In [ ]:
base_model = "models/gemini-1.5-flash-001-tuning"
training_data = []
for i in range(train_neighbor_normalized_df.shape[0]):
    system_p = prompt.finetune_system_deconv(config)
    user_p = prompt.finetune_user_deconv(train_neighbor_normalized_df, i, config)
    assistant_p = prompt.finetune_assistant(train_neighbor_normalized_df, i, adata.obs[config.name_truth])
    training_data.append({'text_input': system_p + user_p, 'output': assistant_p})
    



In [ ]:
training_data[:4]

In [ ]:
operation = genai.create_tuned_model(
    # You can use a tuned model here too. Set `source_model="tunedModels/..."`
    display_name=config.data_name,
    source_model=base_model,
    epoch_count=30,
    batch_size=4,
    learning_rate=0.001,
    training_data=training_data,
)

In [ ]:
for status in operation.wait_bar():
    time.sleep(10)

In [ ]:
result = operation.result()

In [ ]:
import seaborn as sns
model = operation.result()  # model = genai.get_tuned_model()
snapshots = pd.DataFrame(model.tuning_task.snapshots)

sns.lineplot(data=snapshots, x = 'epoch', y='mean_loss')

# Use the finetuned model

## generate json

In [ ]:
# add model name to config
# config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetune-merfish270-3:ARMu34Br"

# add model name to config and reload config
config = load_config("configs/config_finetune_merfish.yaml")
config.data_name = "MERFISH_27"
config.refresh_paths()
name_truth = config.name_truth

unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes

config.system_prompt = prompt.finetune_system_celltype_geneorder(config)




In [ ]:
print(config.system_prompt)

In [ ]:
generate_json_end2end(val_neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, n_rows=1, batch_size=3000, df_extra=val_neighbor_normalized_df_genes)

## submit

In [ ]:

import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetune_merfish.yaml {config.data_name} _rep1 > outs/{config.data_name}_finetune_26.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")
gpt_results_df.columns = ['finetune_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.finetune_gpt4o_mini.value_counts()

In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetune_gpt4o_mini.value_counts()[gpt_results_df.finetune_gpt4o_mini.value_counts()<4].index:
    gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == nichtype, "finetune_gpt4o_mini"] = "unknown"

In [ ]:
gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "MpN", "finetune_gpt4o_mini"] = "MPN"


In [ ]:
# # manually retrieve batch output
# output_file_name = f"{output_path}/response_BZ5_1_{use_full_name}_{with_self_type}_{with_region_name}_{Graph_type}_{with_negatives}_{with_CoT}_{with_count_numbers}.txt"
# batch_id = "batch_66f62020d11c81909424c1423da11a70"
# file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
# # Open the file in write mode and save the string
# with open(output_file_name, 'w') as file:
#     file.write(file_response.text)

## plot and save

In [ ]:
val_adata = adata[val_neighbor_normalized_df.index].copy()
val_adata.obs = val_adata.obs.join(gpt_results_df)
val_adata.obs['finetune_gpt4o_mini'] = val_adata.obs['finetune_gpt4o_mini'].fillna("unknown")
sc.pl.scatter(val_adata, x="x", y="y", color="finetune_gpt4o_mini", title =  f"finetune_gpt4o_mini")

print(adjusted_rand_score(val_adata.obs[config.name_truth], val_adata.obs['finetune_gpt4o_mini']))

In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")



# test

## load test data BZ9 BZ14 
Prototype in prompt is based on training data

In [ ]:
# add model name to config
# config.gpt_model = "ft:gpt-4o-mini-2024-07-18:whh:finetune-merfish270-3:ARMu34Br"

# add model name to config and reload config
config = load_config("configs/config_finetune_merfish.yaml")
config.data_name = "MERFISH_27"
config.refresh_paths()
name_truth = config.name_truth

unique_layers = adata.obs[name_truth].unique()
domain_mapping = {i: layer for i, layer in enumerate(unique_layers)}

# the full cell type names are already in the data
unique_celltypes = adata.obs['cell_type'].unique()
cell_names_mapping = {celltype: celltype for _, celltype in enumerate(unique_celltypes)}

config.domain_mapping = domain_mapping
config.cell_names_mapping = cell_names_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes

config.system_prompt = prompt.finetune_system_celltype_geneorder(config)



In [ ]:
# --- Load data ---
data_path = str(dataset_file("merfish", config.data_name))
adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]

sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)


# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))



In [ ]:
# use the same top_genes as training data

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=top_genes)


## test GPT

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt_func=prompt.finetune_user_celltype_geneorder, n_rows=1, batch_size=3000, df_extra=neighbor_normalized_df_genes)

In [ ]:
config.data_name

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_finetune_merfish.yaml {config.data_name} _rep1 > outs/{config.data_name}_finetune_26.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ").replace("‘", "'").replace("’", "'")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")


In [ ]:
gpt_results_df

In [ ]:
gpt_results_df = gpt_results_df[[0]]
gpt_results_df.columns = ['finetune_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df = pd.read_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.value_counts()

In [ ]:
gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "FX", "finetune_gpt4o_mini"] = "fx"
gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "Mpa", "finetune_gpt4o_mini"] = "MPA"
# gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "'PV", "finetune_gpt4o_mini"] = "PV"
# gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "Outputs: 'PVH'", "finetune_gpt4o_mini"] = "PVH"
gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "Pv", "finetune_gpt4o_mini"] = "PV"
# gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "mpa", "finetune_gpt4o_mini"] = "MPA"
gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "PV.", "finetune_gpt4o_mini"] = "PV"
gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == "pvh", "finetune_gpt4o_mini"] = "PVH"







In [ ]:
# if the number of the cell type is less than 4, set it to unknown
for nichtype in gpt_results_df.finetune_gpt4o_mini.value_counts()[gpt_results_df.finetune_gpt4o_mini.value_counts()<3].index:
    gpt_results_df.loc[gpt_results_df.finetune_gpt4o_mini == nichtype, "finetune_gpt4o_mini"] = "unknown"

## plot and save

In [ ]:
adata.obs.drop(columns=['finetune_gpt4o_mini'], inplace=True)

In [ ]:
adata.obs = adata.obs.join(gpt_results_df)
# replace the NA in adata.obs['finetune_gpt4o'] with "unknown"
adata.obs['finetune_gpt4o_mini'] = adata.obs['finetune_gpt4o_mini'].fillna("unknown")

sc.pl.scatter(adata, x="x", y="y", color="finetune_gpt4o_mini", title =  f"finetune_gpt4o_mini")
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['finetune_gpt4o_mini']))

In [ ]:
sc.pl.scatter(adata, x="x", y="y", color=config.name_truth, title =  f"{config.name_truth}")



In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
print(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
